# Notebook 1: What is an LLM Call?

In this notebook, you'll learn:
- What a Large Language Model (LLM) does: text in → text out
- How to run a model locally using **vLLM**
- How to send prompts and receive completions
- Key parameters: temperature, max_tokens, stop sequences
- How to get structured output (JSON) from a model

## Prerequisites
Before running this notebook, you need:
1. A running vLLM server (instructions below)
2. The `rlm_core` library installed (from this project)

## Step 1: Starting a Local LLM Server

vLLM lets you serve any Hugging Face model as an OpenAI-compatible API.

**vLLM requires Linux.** On Windows, run the server inside **WSL2**:

1. Open a WSL2 terminal
2. Create and activate a Python virtual environment
3. Install vLLM: `pip install vllm`
4. Start the server:

```bash
python -m vllm.entrypoints.openai.api_server \
    --model Orion-zhen/Qwen3-1.7B-AWQ \
    --quantization awq \
    --dtype float16 \
    --max-model-len 2048 \
    --gpu-memory-utilization 0.80 \
    --enforce-eager \
    --port 8000
```

This downloads and serves Qwen3-1.7B (4-bit AWQ quantized, ~1.5GB). Wait until you see "Application startup complete".

**What just happened?**
- vLLM loaded a **language model** — a neural network trained to predict the next word
- It's now serving it as an API on your local machine
- We can talk to it just like OpenAI's API, but everything runs on YOUR GPU
- WSL2 forwards port 8000 to Windows automatically

## Step 2: Your First LLM Call

An LLM is fundamentally simple: you give it text, it gives you text back.

Let's use our `LLMClient` wrapper to make this easy:

In [ ]:
import sys
sys.path.insert(0, "..")  # So we can import rlm_core

from rlm_core import LLMClient

# Connect to our local vLLM server
client = LLMClient(
    base_url="http://localhost:8000/v1",
    model="Orion-zhen/Qwen3-1.7B-AWQ"
)

# Make our first call!
result = client.completion("What is 2 + 2?")
print("Model says:", result.text)
print(f"\nTokens used: {result.prompt_tokens} prompt + {result.completion_tokens} completion")

## Understanding What Happened

1. We sent the text `"What is 2 + 2?"` to the model
2. The model **tokenized** it — broke it into small pieces called tokens
3. For each token position, the model predicted the most likely **next token**
4. It kept generating tokens until it decided to stop
5. We got back text + token counts

**Tokens** are the currency of LLMs. Every call costs tokens. This matters when we build RLMs later — recursive calls multiply token usage.

## Step 3: System Prompts — Giving the Model a Role

A **system prompt** tells the model WHO it should be. The user prompt tells it WHAT to do.

In [ ]:
# Without a system prompt — the model decides its own personality
result1 = client.completion("Explain gravity in one sentence.")
print("Default:", result1.text)

print("\n---\n")

# With a system prompt — we control the style
result2 = client.completion(
    "Explain gravity in one sentence.",
    system="You are a pirate who explains science. Use pirate language."
)
print("Pirate:", result2.text)

## Step 4: Temperature — Controlling Randomness

**Temperature** controls how "creative" vs "deterministic" the model is:
- `temperature=0.0` → Always picks the most likely token (deterministic)
- `temperature=0.7` → Balanced (default)
- `temperature=1.5` → Very creative/random

Let's see the difference:

In [ ]:
prompt = "Write a one-sentence story about a cat."

print("=== Temperature 0.0 (deterministic) ===")
for i in range(3):
    r = client.completion(prompt, temperature=0.0, max_tokens=50)
    print(f"  Run {i+1}: {r.text.strip()}")

print("\n=== Temperature 1.2 (creative) ===")
for i in range(3):
    r = client.completion(prompt, temperature=1.2, max_tokens=50)
    print(f"  Run {i+1}: {r.text.strip()}")

## Step 5: Stop Sequences — Controlling When the Model Stops

By default, the model generates until `max_tokens`. But you can tell it to **stop early** when it produces certain text. This is critical for RLMs — we use stop sequences to detect when the model has finished writing code.

In [ ]:
# Without stop sequence — model rambles
result = client.completion(
    "List 3 colors, one per line.",
    max_tokens=100,
    temperature=0.0
)
print("Without stop:", repr(result.text))

print("\n---\n")

# With stop sequence — stops after 3 items
result = client.completion(
    "List 3 colors, one per line. Start with '1.'",
    max_tokens=100,
    temperature=0.0,
    stop=["\n4."]  # Stop before a 4th item
)
print("With stop:", repr(result.text))

## Step 6: Structured Output — Getting JSON from the Model

For RLMs, the model needs to output **code**, not prose. Let's practice getting structured output:

In [ ]:
result = client.completion(
    "List 3 programming languages with their year of creation. "
    "Respond ONLY with a JSON array, no other text. Example format: "
    '[{"name": "Python", "year": 1991}]',
    temperature=0.0
)

print("Raw response:", result.text)

import json
try:
    data = json.loads(result.text)
    print("\nParsed successfully!")
    for lang in data:
        print(f"  {lang['name']}: {lang['year']}")
except json.JSONDecodeError as e:
    print(f"\nFailed to parse JSON: {e}")
    print("This is normal — models don't always produce valid JSON!")

## Step 7: Token Tracking — Measuring Cost

Every LLM call costs tokens. Our `LLMClient` tracks cumulative usage:

In [ ]:
print(f"Total calls so far: {client.call_count}")
print(f"Total prompt tokens: {client.total_prompt_tokens}")
print(f"Total completion tokens: {client.total_completion_tokens}")
print(f"Total tokens: {client.total_prompt_tokens + client.total_completion_tokens}")

## Key Takeaways

1. **LLMs are text-to-text functions** — they take text input and produce text output
2. **System prompts** control the model's behavior and personality
3. **Temperature** controls randomness — low = deterministic, high = creative
4. **Stop sequences** let you control when generation stops
5. **Tokens** are the unit of cost — every call uses them
6. **Structured output** (JSON, code) is possible but not guaranteed

## What's Next?

In Notebook 2, we'll build a **sandbox** that can execute Python code generated by the LLM. This is the foundation of how RLMs work — the model doesn't just generate text, it generates **programs** that manipulate data.